# Data preparation

This script was developed to bring in relevant environmental data for modeling carbon stock as a function of environmental co-variates from remote sensing and Copernicus data products.

Note: 
* The first function (extract_values) extracts values of matching sites, however this gives NAs for sites that appear over land given the lower resolution data products.
* The second function extract_closest_values() provides the closest match for any sites that received an NA.

## Input data:

* Seagrass site data, `Seagrass_site`, Seagrass_site_data.xlsx
* 95th percentile of bottom temperature (°C): `bottomT_p95_C_closest`, bottomT_p95_daily_C.nc
* Eastward seawater velocity (m s-1), `uo_mean_1.5m_m_s_closest`, uo_mean_1.5m_m_s.nc
* Northward seawater velocity (m s-1), `vo_p90_1.5m_m_s_closest`, vo_p90_1.5m_m_s.nc
* Phosphate at sub surface depth 1.5 m (mmol m-3), `po4_mean_1.5m_mmol_m3`, po4_mean_monthly_1.5m_mmol_m3.nc
* PH at sub surface depth 1.5 m (1), `pH_mean_1.5m`, pH_mean_monthly_1.5m.nc
* Sea surface wave significant height (m), `wave_height_VHM0_p95_m`, VHM0_p95_m.nc
* Surface downward flux of total CO2 (molC m-2 yr-1), `Surf_fgco2_p95_molC_m2_yr`, Surf_fgco2_p95_molC_m2_yr.nc
* Diffuse attenuation coefficient at 490 nm, `KD490`, S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc
* Remote sensing reflectance at 443 nm, `RRS443`, S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km.nc


## Bottom_T_p95

Data comes from the Global Ocean Physics Reanalysis.
I used the daily product for bottom_T and the monthly product for the current data, [GLOBAL_MULTIYEAR_PHY_001_030](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services)

* Input: `cmems_mod_glo_phy_my_0.083deg_P1D-m`, GLOBAL_MULTIYEAR_PHY_001_030, "bottomT" is Sea water potential temperature at sea floorbottomT [°C]
* Output: `bottomT_p95_daily_C.nc` is 95th percentile of bottom temperature at each location from daily physics product

### Load NetCDF

```R
bottomT_daily <- open.nc("data/cmems_mod_glo_phy_my_0.083deg_P1D-m_bottomT_10.00W-34.00E_34.00N-61.00N_1993-01-01-2021-06-30.nc")
print.nc(bottomT_daily)
```

### Load NetCDF as SpatRaster

```R
bottomT_daily_raster <- rast("data/cmems_mod_glo_phy_my_0.083deg_P1D-m_bottomT_10.00W-34.00E_34.00N-61.00N_1993-01-01-2021-06-30.nc", subds = "bottomT")

# Calculate the 95th percentile of bottom temperature at each location from the daily physics product
p95_bottomT_daily_C <- app(bottomT_daily_raster, fun = function(x) unname(quantile(x, probs = 0.95, na.rm = TRUE)))
plot(p95_bottomT_daily_C)

# Write this to a NetCDF for future use
writeCDF(p95_bottomT_daily_C, "bottomT_p95_daily_C.nc", varname = "p95_bottomT_daily_C", overwrite = TRUE)
```

## Uo_mean

Eastward seawater velocity (m s-1)

Note: uo(time, depth, latitude, longitude) float64 8GB dask.array<chunksize=(68, 2, 64, 64), meta=np.ndarray>

* Input: `cmems_mod_glo_phy_my_0.083deg_P1M-m`
* Output: `uo_mean_1.5m_m_s.nc`

### Load NetCDF

```R
physics_monthly <- open.nc("data/cmems_mod_glo_phy_my_0.083deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.49-47.37m_1993-01-01-2021-06-01.nc")
print.nc(physics_monthly)
```

### Load NetCDF as SpatRaster

```R
uo_monthly <- rast("data/cmems_mod_glo_phy_my_0.083deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.49-47.37m_1993-01-01-2021-06-01.nc", subds = "uo")
uo_monthly_1.5m <- uo_monthly[[grep("1.54", names(uo_monthly))]]
```

### calculate mean across time

```R
uo_mean_1.5m <- app(uo_monthly_1.5m, fun = mean, na.rm = TRUE)
plot(uo_mean_1.5m)
writeCDF(uo_mean_1.5m, "uo_mean_1.5m_m_s.nc", varname = "uo_mean_1.5m_m_s", overwrite = TRUE)
```

## Vo_p90 

Northward seawater velocity (m s-1)

Note: vo(time, depth, latitude, longitude) float64 8GB dask.array<chunksize=(68, 2, 64, 64), meta=np.ndarray>

* Input: `cmems_mod_glo_phy_my_0.083deg_P1M-m`
* Output: `vo_p90_1.5m_m_s.nc`

### Load NetCDF as SpatRaster

```R
vo_monthly <- rast("data/cmems_mod_glo_phy_my_0.083deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.49-47.37m_1993-01-01-2021-06-01.nc", subds = "vo")
vo_monthly_1.5m <- vo_monthly[[grep("1.54", names(vo_monthly))]]
```

### Now apply the 90th percentile across time (i.e., across these layers)

```R
vo_p90_1.5m <- app(vo_monthly_1.5m, fun = function(x) unname(quantile(x, probs = 0.90, na.rm = TRUE)))

plot(vo_p90_1.5m)
writeCDF(vo_p90_1.5m, "vo_p90_1.5m_m_s.nc", varname = "vo_p90_1.5m_m_s", overwrite = TRUE)
```

## Phosphate_mean

Data comes from the Global Ocean Biogeochemistry Hindcast monthly product,
[GLOBAL_MULTIYEAR_BGC_001_029](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_BGC_001_029/services)

Will select the sub surface depth 1.55585503578186 m

Note: NC_FLOAT po4(longitude, latitude, depth, time); NC_CHAR po4:long_name = "Phosphate"; NC_CHAR po4:units = "mmol m-3";

* Input: `cmems_mod_glo_bgc_my_0.25deg_P1M-m`
* Output: `po4_mean_1.5m.nc`

### Load NetCDF

```R
BGC_monthly <- open.nc("data/cmems_mod_glo_bgc_my_0.25deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.51-1.56m_1993-01-01-2022-12-01.nc")
print.nc(BGC_monthly) # Print report
```

### Load NetCDF as SpatRaster

```R
po4_monthly <- rast("data/cmems_mod_glo_bgc_my_0.25deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.51-1.56m_1993-01-01-2022-12-01.nc", subds = "po4")
po4_monthly_1.5m <- po4_monthly[[grep("1.5558", names(po4_monthly))]]

# calculate mean across time
po4_mean_1.5m <- app(po4_monthly_1.5m, fun = mean, na.rm = TRUE)
plot(po4_mean_1.5m)
writeCDF(po4_mean_1.5m, "po4_mean_monthly_1.5m_mmol_m3.nc", varname = "po4_mean_1.5m_mmol_m3", overwrite = TRUE)
```

## pH_mean

Data comes from the Global Ocean Biogeochemistry Hindcast monthly product,
[GLOBAL_MULTIYEAR_BGC_001_029](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_BGC_001_029/services)

Note: NC_FLOAT ph(longitude, latitude, depth, time); NC_CHAR ph:long_name = "PH"; NC_CHAR ph:units = "1";

* Input: `cmems_mod_glo_bgc_my_0.25deg_P1M-m`
* Output: `pH_mean_1.5m.nc`

### Load NetCDF as SpatRaster

```R
pH_monthly <- rast("data/cmems_mod_glo_bgc_my_0.25deg_P1M-m_multi-vars_9.00W-34.00E_34.00N-61.00N_0.51-1.56m_1993-01-01-2022-12-01.nc", subds = "ph")
pH_monthly_1.5m <- pH_monthly[[grep("1.5558", names(pH_monthly))]]

# calculate mean across time
pH_mean_1.5m <- app(pH_monthly_1.5m, fun = mean, na.rm = TRUE)
plot(pH_mean_1.5m)
writeCDF(pH_mean_1.5m, "pH_mean_monthly_1.5m.nc", varname = "pH_mean_1.5m", overwrite = TRUE)
```

## VHM0_p95

Global Ocean Waves Reanalysis.
3-hour product, 0.2 degrees, only selecting Sea surface wave significant height VHM0 [m] (1980-2023), [GLOBAL_MULTIYEAR_WAV_001_032](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_WAV_001_032/download?dataset=cmems_mod_glo_wav_my_0.2deg_PT3H-i_202411)

* Input: `cmems_mod_glo_wav_my_0.2deg_PT3H-i`
* Output: `VHM0_p95_m.nc`

### Load NetCDF

```R
Wave_height <- open.nc("data/cmems_mod_glo_wav_my_0.2deg_PT3H-i_VHM0_9.00W-34.00E_34.00N-61.00N_1980-01-01-2023-04-30.nc")
print.nc(Wave_height)
```

### Load NetCDF as SpatRaster

```R
Wave_height_rast <- rast("data/cmems_mod_glo_wav_my_0.2deg_PT3H-i_VHM0_9.00W-34.00E_34.00N-61.00N_1980-01-01-2023-04-30.nc")

# calculate the 95th percentile  across time (This is a slow step)
VHM0_p95_m <- app(Wave_height_rast, fun = function(x) unname(quantile(x, probs = 0.95, na.rm = TRUE)))
plot(VHM0_p95_m)
writeCDF(VHM0_p95_m, "wave_height_p95_m.nc", varname = "wave_height_VHM0_p95_m", overwrite = TRUE)
```

## fgCO2_p95

Surface ocean carbon fields, [MULTIOBS_GLO_BIO_CARBON_SURFACE_MYNRT_015_008](https://data.marine.copernicus.eu/product/MULTIOBS_GLO_BIO_CARBON_SURFACE_MYNRT_015_008/services).
1985-2023 (there is also a near-real time 2024-2025 product), monthly product

Note: NC_FLOAT fgco2(longitude, latitude, time); NC_FLOAT fgco2:_FillValue = 9.96920996838687e+36; NC_STRING fgco2:long_name = "Surface downward flux of total CO2"; NC_STRING fgco2:units = "molC m-2 yr-1"; NC_STRING fgco2:standard_name = "surface_downward_mass_flux_of_carbon_dioxide_expressed_as_carbon";

* Input: `cmems_obs-mob_glo_bgc-car_my_irr-i`
* Output: `Surf_fgco2_p95_molC_m2_yr.nc`

### Load NetCDF

```R
Carbon <- open.nc("data/cmems_obs-mob_glo_bgc-car_my_irr-i_multi-vars_8.88W-33.88E_34.12N-60.88N_1985-01-01-2023-12-01.nc")
print.nc(Carbon)
```

### Load NetCDF as SpatRaster

```R
Surf_fgco2_rast <- rast("data/cmems_obs-mob_glo_bgc-car_my_irr-i_multi-vars_8.88W-33.88E_34.12N-60.88N_1985-01-01-2023-12-01.nc", subds = "fgco2")

# calculate the 95th percentile  across time
Surf_fgco2_p95_molC_m2_yr <- app(Surf_fgco2_rast, fun = function(x) unname(quantile(x, probs = 0.95, na.rm = TRUE)))
writeCDF(Surf_fgco2_p95_molC_m2_yr, "Surf_fgco2_p95_molC_m2_yr.nc", varname = "Surf_fgco2_p95_molC_m2_yr", overwrite = TRUE)
```

## KD490

Diffuse attenuation coefficient at 490 nm (Entire mission composite) (4km) (Standard) (S3A-OLCI).
Data was downloaded from [oceandata.sci.gsfc.nasa.gov](https://oceandata.sci.gsfc.nasa.gov/l3/order/) on March 11, 2025,
(Entire mission composite) (4km) (Standard) (S3A-OLCI).

This parameter is widely used in oceanography to assess water clarity and the presence of particles in the water column.
A lower Kd490 value indicates clearer water, as light can penetrate deeper.
Conversely, a higher value suggests more turbidity due to particles like phytoplankton, sediments, or organic matter.

Note: that The S3A-OLCI satellite offers high-resolution data (300 m spatial resolution), but these data products are 4km

* Input: `S3A_OLCI_ERRNT`
* Output: `S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc`

## RRS443

Remote sensing reflectance at 443 nm (Entire mission composite) (4km) (Standard) (S3A-OLCI).
Generally close to chla product.
Data was downloaded from [oceandata.sci.gsfc.nasa.gov](https://oceandata.sci.gsfc.nasa.gov/l3/order/) on March 11, 2025,
(Entire mission composite) (4km) (Standard) (S3A-OLCI).

Note: that The S3A-OLCI satellite offers high-resolution data (300 m spatial resolution), but these data products are 4km

* Input: `S3A_OLCI_ERRNT`
* Output: `S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc`

# Carbon stock prediction

This script was developed to predict carbon stock in the upper 30 and 100 cm of the sediment of a seagrass bed based on the dataset produced in the script "Environmental_covariates_matching_VRE.R", called "SG_modeling_dataframe.csv"

# Input

* `file_SG_modeling_dataframe`, SG_modeling_dataframe.csv, prepared in `T5.3_step01-Env_cov_matching.ipynb`.
* `file_For_modeling_df_shallow_carbon_density`, For_modeling_df_shallow_carbon_density.rds, from [EURO-CARBON](https://zenodo.org/records/14905489)
* `file_GAM_top_reduced_SGstock`, GAM_top_reduced_model_SGstock.rds, carbon density prediction model (prepared in script: `SG_carbonstock_model.R`)

# calculate carbon stock

Carbon stock = Carbon density (gC/cm3) x Depth (cm) x 10,000

Note: 
* 1 hectare = 10,000 square metres (m²), Equivalent to 2.471 acres
* Carbon stocks are often expressed as: tonnes of carbon per hectare (t C/ha) OR megagrams of carbon per hectare (Mg C/ha) (1 Mg = 1 tonne)
* Carbon density is in: gC cm-3